Caso 1

In [1]:
# Funciones auxiliares — Caso 1: Árbol de regresión (Car Price)
# Son las mismas que usamos en el laboratorio de árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_regresion ahora también reporta el MAPE (%).
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)

In [2]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/CarPrice_dataset.csv')

# Mostrar las primeras filas
print("Muestra del dataset:")
df.head()

Muestra del dataset:


,CarName,carbody,fueltype,enginetype,cylindernumber,doornumber,symboling,wheelbase,carlength,carwidth,...,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,alfa-romero giulia,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,13495.0
1,alfa-romero stelvio,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,16500.0
2,alfa-romero Quadrifoglio,hatchback,gas,ohcv,6,2,1,94.5,171.2,65.5,...,2823,152,2.68,3.47,9.0,154,5000,19,26,16500.0
3,audi 100 ls,sedan,gas,ohc,4,4,2,99.8,176.6,66.2,...,2337,109,3.19,3.40,10.0,102,5500,24,30,13950.0
4,audi 100ls,sedan,gas,ohc,5,4,2,99.4,176.6,66.4,...,2824,136,3.19,3.40,8.0,115,5500,18,22,17450.0


symboling 

Its assigned insurance risk rating, A value of +3 indicates that the auto is risky, -3 that it is probably pretty safe. (Numeric) 

fueltype

Car fuel type i.e gas or diesel (Categorical)

doornumber

Number of doors in a car (Numeric)

carbody

body of car (Categorical)

wheelbase

Wheelbase of car (Numeric)

carlength

Length of car (Numeric)

carwidth

Width of car (Numeric)

carheight

height of car (Numeric)

curbweight

The weight of a car without occupants or baggage. (Numeric)

enginetype

Type of engine. (Categorical)

cylindernumber

cylinder placed in the car (Numeric)

enginesize

Size of the engine (Numeric)

boreratio

Boreratio of car (Numeric)

stroke

Stroke or volume inside the engine (Numeric)

compressionratio

compression ratio of car (Numeric)

horsepower

Horsepower (Numeric)

peakrpm

car peak rpm (Numeric)

citympg

Mileage in city (Numeric)

highwaympg

Mileage on highway (Numeric)

price (Dependent variable)

Price of car (Numeric) in dollars

La compañía desea que realices un análisis preciso y muy rápido. Por tanto, tú sabes que lo mejor será realizar todo el análisis únicamente usando Decision Trees Regressor, ya que es el mejor modelo que conoces. Ten en cuenta que no es necesario transformar las variables categóricas a categóricas numéricas, usa todas las columnas numéricas del DataFrame (select_dtypes), tampoco es necesario escalarlas.

Parte 1. Feature Selection [Valor 8 puntos].

Antes de responder, realiza los siguientes pasos en tu notebook:

Divide el conjunto en entrenamiento y prueba usando train_test_split con los parámetros test_size=0.2 y random_state=0.
Entrena un modelo sencillo de DecisionTreeRegressor con random_state=0.
Obtén las importancias de las variables y grafícalas en un gráfico de barras horizontales.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

def preparar_car(df, test_size=0.2, random_state=0):
    """Usa solo columnas numéricas y hace split 80/20."""
    df_num = df.select_dtypes(include='number')
    X = df_num.drop(columns=['price'])
    y = df_num['price']

    print("Columnas usadas:", list(X.columns))
    print("Shape X:", X.shape)

    return train_test_split(X, y, test_size=test_size, random_state=random_state)


X_train, X_test, y_train, y_test = preparar_car(df)

Columnas usadas: ['cylindernumber', 'doornumber', 'symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg']
Shape X: (205, 16)


In [4]:
arbol_sin_podar = DecisionTreeRegressor(random_state=0)
arbol_sin_podar.fit(X_train, y_train)

print("R2 en train:", arbol_sin_podar.score(X_train, y_train))
print("R2 en test:", arbol_sin_podar.score(X_test, y_test))
print("Número de hojas:", arbol_sin_podar.get_n_leaves())
print("Profundidad:", arbol_sin_podar.get_depth())

R2 en train: 0.9984642634434308
R2 en test: 0.8589222097924514
Número de hojas: 150
Profundidad: 14


In [5]:
# Importancias
importancias = plot_importancias(arbol_sin_podar, X_train.columns)
print(importancias)

enginesize          0.675420
curbweight          0.235578
carwidth            0.019698
compressionratio    0.016401
peakrpm             0.015779
horsepower          0.011656
stroke              0.006883
carlength           0.005857
symboling           0.004067
citympg             0.003412
wheelbase           0.002212
highwaympg          0.001147
boreratio           0.000769
carheight           0.000670
doornumber          0.000377
cylindernumber      0.000073
dtype: float64


In [6]:
cols = ['enginesize', 'curbweight']
X = df[cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

arbol_80 = DecisionTreeRegressor(random_state=0)
arbol_80.fit(X_train, y_train)

print("R2 train:", round(arbol_80.score(X_train, y_train), 4))
print("R2 test:", round(arbol_80.score(X_test, y_test), 4))

print("X_train.shape =", X_train.shape)
print("X_test.shape =", X_test.shape)
print("y_train.shape =", y_train.shape)
print("y_test.shape =", y_test.shape)

R2 train: 0.9983
R2 test: 0.8886
X_train.shape = (164, 2)
X_test.shape = (41, 2)
y_train.shape = (164,)
y_test.shape = (41,)


In [7]:

pred_train = arbol_80.predict(X_train)
pred_test = arbol_80.predict(X_test)

mape_train = mean_absolute_percentage_error(y_train, pred_train) * 100
mape_test = mean_absolute_percentage_error(y_test, pred_test) * 100
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print(f"MAPE_train = {mape_train:.2f}")
print(f"MAPE_test = {mape_test:.2f}")
print(f"RMSE_train = {rmse_train:.2f}")
print(f"RMSE_test = {rmse_test:.2f}")

MAPE_train = 0.77
MAPE_test = 18.51
RMSE_train = 322.88
RMSE_test = 2936.83


Caso 2

In [8]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1.

    Con más de dos clases, precisión, recall y F1 son el promedio simple
    de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos,
            p. ej. ['Low', 'Mid_Low', 'Mid', 'High']. Si no las pasas, se ordenan alfabéticamente.
    etiquetas: nombres a mostrar para esas clases (opcional; por defecto, las mismas clases).
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,px_height,px_width,ram,talk_time,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,20,756,2549,19,0,1,Mid_Low
1,1021,1,0.5,1,0,1,53,0.7,136,3,905,1988,2631,7,1,0,Mid
2,563,1,0.5,1,2,1,41,0.9,145,5,1263,1716,2603,9,1,0,Mid
3,615,1,2.5,0,0,0,10,0.8,131,6,1216,1786,2769,11,0,0,Mid
4,1821,1,1.2,0,13,1,44,0.6,141,2,1208,1212,1411,15,1,0,Mid_Low


In [10]:

orden = ['Low', 'Mid_Low', 'Mid', 'High']

conteo = (df['price_range']
          .value_counts()
          .reindex(orden)
          .rename_axis('price_range')
          .reset_index(name='Observaciones'))

fig = px.bar(conteo, x='price_range', y='Observaciones',
             text='Observaciones', color='price_range',
             category_orders={'price_range': orden},
             title='Cantidad de observaciones por price_range',
             template='plotly_white')
fig.update_layout(showlegend=False, xaxis_title='Rango de precio',
                  yaxis_title='Cantidad de observaciones')
fig.show()

In [11]:
orden = {'Low': 0, 'Mid_Low': 1, 'Mid': 2, 'High': 3}
df_corr = df.copy()
df_corr['price_range'] = df_corr['price_range'].map(orden)

corr_target = (df_corr.select_dtypes(include='number')
               .corr()['price_range']
               .drop('price_range'))

# Ordenado por fuerza de la relación (valor absoluto)
print(corr_target.reindex(corr_target.abs().sort_values(ascending=False).index).round(4))

print("\nMayor relación:", corr_target.abs().idxmax())
print("Menor relación:", corr_target.abs().idxmin())

# Mapa de calor con tu función
#plot_correlation_matrix(df_corr, df_corr.select_dtypes(include='number').columns)

ram              0.9216
battery_power    0.2069
px_width         0.1715
px_height        0.1560
int_memory       0.0474
blue             0.0314
mobile_wt       -0.0308
touch_screen    -0.0299
fc               0.0294
talk_time        0.0253
wifi             0.0214
dual_sim         0.0165
four_g           0.0155
m_dep            0.0049
n_cores          0.0026
clock_speed     -0.0021
Name: price_range, dtype: float64

Mayor relación: ram
Menor relación: clock_speed


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV

# 1. Split (sin stratify)
X = df.drop(columns=['price_range'])
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# 2. Escalado ajustado solo con train
X_train_s, X_test_s = escalar(X_train, X_test)

# 3. Regresión logística + RFECV
log_reg = LogisticRegression(random_state=0, max_iter=500)
rfecv = RFECV(estimator=log_reg, cv=5)
rfecv.fit(X_train_s, y_train)

print("Número óptimo de variables:", rfecv.n_features_)
print("Variables seleccionadas:", list(X_train_s.columns[rfecv.support_]))

plot_rfecv_clasificacion(rfecv, metrica='exactitud')

Número óptimo de variables: 5
Variables seleccionadas: ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']


In [13]:

sel = list(X_train_s.columns[rfecv.support_])

# Matriz de correlación con tu función
plot_correlation_matrix(X_train_s, sel)

# Revisión automática: pares con |r| > 0.8
corr = X_train_s[sel].corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pares = corr.where(mask).stack()

print("Correlación máxima (abs):", round(pares.abs().max(), 4))
print("Pares con |r| > 0.8:\n", pares[pares.abs() > 0.8])
print("\nMulticolinealidad:", (pares.abs() > 0.8).any())

Correlación máxima (abs): 0.4947
Pares con |r| > 0.8:
 Series([], dtype: float64)

Multicolinealidad: False


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Mismo split que en RFECV (sin stratify)
X = df.drop(columns=['price_range'])
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# Mismo escalamiento: ajustado solo con train y con todas las variables
X_train_s, X_test_s = escalar(X_train, X_test)

# Variables seleccionadas por RFECV
sel = ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']
X_train_sel = X_train_s[sel]
X_test_sel = X_test_s[sel]

# Modelo
log_sel = LogisticRegression(random_state=0, max_iter=500)
log_sel.fit(X_train_sel, y_train)

pred_train = log_sel.predict(X_train_sel)
pred_test = log_sel.predict(X_test_sel)

# Métricas (4 decimales, escala 0 a 1; macro por ser 4 clases)
m_train = evaluar_clasificador('LogReg 5 vars - train', y_train, pred_train)
m_test = evaluar_clasificador('LogReg 5 vars - test', y_test, pred_test)

# Matriz de confusión en test
cm = plot_matriz_confusion(y_test, pred_test,
                           clases=['Low', 'Mid_Low', 'Mid', 'High'],
                           title='Matriz de confusión - test')
print(cm)

LogReg 5 vars - train: exactitud 0.9723 | precisión 0.9724 | recall 0.9706 | F1 0.9713
LogReg 5 vars - test: exactitud 0.9676 | precisión 0.9677 | recall 0.9661 | F1 0.9666


               Predicho: Low  Predicho: Mid_Low  Predicho: Mid  Predicho: High
Real: Low                 93                  0              0               0
Real: Mid_Low              3                 76              1               0
Real: Mid                  0                  3             85               4
Real: High                 0                  0              1             104


In [15]:
from sklearn.metrics import accuracy_score

acc_train = accuracy_score(y_train, pred_train)
acc_test = accuracy_score(y_test, pred_test)
dif = acc_train - acc_test

print(f"Accuracy_train = {acc_train:.4f}")
print(f"Accuracy_test = {acc_test:.4f}")
print(f"Diferencia = {dif:.4f}")

Accuracy_train = 0.9723
Accuracy_test = 0.9676
Diferencia = 0.0047


In [16]:
# Matriz de confusión en test (modelo con las 5 variables de RFECV)
cm = plot_matriz_confusion(
    y_test, pred_test,
    clases=['Low', 'Mid_Low', 'Mid', 'High'],
    title='Matriz de confusión - Regresión Logística (variables RFECV)'
)
cm

,Predicho: Low,Predicho: Mid_Low,Predicho: Mid,Predicho: High
Real: Low,93,0,0,0
Real: Mid_Low,3,76,1,0
Real: Mid,0,3,85,4
Real: High,0,0,1,104
